# mzero: Zero-Configuration RAG Framework in Google Colab

Welcome to the interactive Google Colab notebook for **mzero**.

This notebook demonstrates:
1. Installing `mzero` and its dependencies
2. Initializing a document knowledge base
3. Running synchronous and asynchronous RAG queries
4. Streaming tokens in real-time
5. Launching an interactive Streamlit UI directly from Colab via `localtunnel`
6. Customizing the pipeline configuration (chunkers, embedders, rerankers, caches)

## 1. Install mzero & Optional Dependencies

In [ ]:
# Install mzero enterprise bundle with full framework support
!pip install -q mzero[full] nest_asyncio

## 2. Create Sample Knowledge Base Documents

We will create a `./docs` folder containing sample policy and documentation files.

In [ ]:
import os

# Create docs directory
os.makedirs('./docs', exist_ok=True)

# Create a sample document
with open('./docs/company_policy.txt', 'w') as f:
    f.write('''
    mzero Enterprise Knowledge Base Policy

    1. Return & Refund Policy:
       Customers are eligible for a 100% full refund within 30 days of purchase.
       Products must be returned in their original packaging with proof of purchase.

    2. Warranty & Support:
       All hardware devices include a 2-year full manufacturer warranty.
       Dedicated 24/7 technical support is available at support@mzero.ai.

    3. API Usage Limits:
       Standard tier provides up to 10,000 queries per month.
       Enterprise tier offers unlimited queries with dedicated SLAs.
    ''')

print('Created sample document in ./docs/company_policy.txt')

## 3. Synchronous RAG Querying

Point `mzero` to your `./docs` directory with a single line of code.

In [ ]:
from mzero import RAG

# Initialize pipeline (automatically indexes documents in ./docs)
rag = RAG(docs_path='./docs')

# Ask a question
response = rag.ask('What is the refund period and warranty duration?')

print('Answer:')
print(response.answer)
print('\nConfidence Score:', response.confidence)

print('\nCitations:')
for cite in response.citations:
    print(f'- {cite.source_file}: "{cite.snippet}"')


## 4. Real-time Token Streaming

Stream tokens directly as they are generated by the pipeline.

In [ ]:
from mzero import RAG

rag = RAG(docs_path='./docs')

print('Streaming Answer: ', end='')
for token in rag.stream('How do I contact technical support?'):
    print(token, end='', flush=True)
print()


## 5. Asynchronous SDK Execution

Use `AsyncRAG` for non-blocking asynchronous operations inside notebook environments.

In [ ]:
import asyncio
import nest_asyncio
from mzero import AsyncRAG

# Enable async nested event loops in Colab
nest_asyncio.apply()

async def run_async_query():
    rag = AsyncRAG(docs_path='./docs')
    res = await rag.aask('What are the API rate limits?')
    print('Async Answer:', res.answer)

await run_async_query()


## 6. Expose Interactive Streamlit Chat UI in Colab

Launch the `mzero` Streamlit adapter and serve it using `localtunnel`.

In [ ]:
# Write Streamlit app file
with open('app.py', 'w') as f:
    f.write('''
import streamlit as st
from mzero.adapters.streamlit import render_mzero_chat

st.set_page_config(page_title="mzero Colab Chat UI", layout="wide")
render_mzero_chat(docs_path="./docs")
''')

print('Wrote app.py successfully.')

In [ ]:
# Get IP for localtunnel authentication password
!curl -s https://loca.lt/mytempip
print('\nCopy the IP above, then click the localtunnel URL below:')

# Start Streamlit and tunnel
!streamlit run app.py & npx localtunnel --port 8501

## 7. Custom Pipeline Configuration & Low-Level API

Override default models, vector databases, and retrieval strategies using `Config`.

In [ ]:
from mzero.config import Config
from mzero.core.pipeline import RAGPipeline

# Create custom pipeline configuration
config = Config(
    docs_path='./docs',
    embedding_model='bge-small-en-v1.5',
    vector_db_backend='faiss',
    top_k=5,
    enable_reranker=True,
    enable_semantic_cache=True
)

# Initialize pipeline
pipeline = RAGPipeline(config)

# Query pipeline
result = pipeline.query('Explain the warranty terms')
print('Custom Pipeline Answer:', result.answer)